In [1]:
from docetl.api import Pipeline, Dataset, MapOp, UnnestOp, ResolveOp, ReduceOp, PipelineStep, PipelineOutput

# Define the dataset - JSON file with medical transcripts
dataset = Dataset(
    type="file",
    path="./medical_transcripts.json"
)

# Define operations
operations = [
    # Extract medications from each transcript
    MapOp(
        name="extract_medications",
        type="map",
        prompt="""
        Analyze the following transcript of a conversation between a doctor and a patient:
        {{ input.src }}
        Extract and list all medications mentioned in the transcript.
        If no medications are mentioned, return an empty list.
        """,
        output={
            "schema": {
                "medication": "list[str]"
            }
        }
    ),
    # Unnest to create separate items for each medication
    UnnestOp(
        name="unnest_medications",
        type="unnest",
        unnest_key="medication"
    ),
    # Resolve similar medication names
    ResolveOp(
        name="resolve_medications",
        type="resolve",
        blocking_keys=["medication"],
        blocking_threshold=0.6162,
        comparison_prompt="""
        Compare the following two medication entries:
        Entry 1: {{ input1.medication }}
        Entry 2: {{ input2.medication }}
        Are these medications likely to be the same or closely related?
        """,
        embedding_model="text-embedding-3-small",
        output={
            "schema": {
                "medication": "str"
            }
        },
        resolution_prompt="""
        Given the following matched medication entries:
        {% for entry in inputs %}
        Entry {{ loop.index }}: {{ entry.medication }}
        {% endfor %}
        Determine the best resolved medication name for this group of entries. The resolved
        name should be a standardized, widely recognized medication name that best represents
        all matched entries.
        """
    ),
    # Summarize side effects and uses for each medication
    ReduceOp(
        name="summarize_prescriptions",
        type="reduce",
        reduce_key=["medication"],
        prompt="""
        Here are some transcripts of conversations between a doctor and a patient:

        {% for value in inputs %}
        Transcript {{ loop.index }}:
        {{ value.src }}
        {% endfor %}

        For the medication {{ reduce_key }}, please provide the following information based on all the transcripts above:

        1. Side Effects: Summarize all mentioned side effects of {{ reduce_key }}.
        2. Therapeutic Uses: Explain the medical conditions or symptoms for which {{ reduce_key }} was prescribed or recommended.

        Ensure your summary:
        - Is based solely on information from the provided transcripts
        - Focuses only on {{ reduce_key }}, not other medications
        - Includes relevant details from all transcripts
        - Is clear and concise
        - Includes quotes from the transcripts
        """,
        output={
            "schema": {
                "side_effects": "str",
                "uses": "str"
            }
        }
    )
]

# Define pipeline step
step = PipelineStep(
    name="medical_info_extraction",
    input="transcripts",
    operations=[
        "extract_medications",
        "unnest_medications",
        "resolve_medications",
        "summarize_prescriptions"
    ]
)

# Define output
output = PipelineOutput(
    type="file",
    path="./medication_summaries.json",
    intermediate_dir="intermediate_results"
)

# Define system prompt (optional but recommended)
system_prompt = {
    "dataset_description": "a collection of transcripts of doctor visits",
    "persona": "a medical practitioner analyzing patient symptoms and reactions to medications"
}

# Create the pipeline
pipeline = Pipeline(
    name="medical_transcript_analysis",
    datasets={"transcripts": dataset},
    operations=operations,
    steps=[step],
    output=output,
    default_model="gpt-4o-mini",
    system_prompt=system_prompt
)

# Run the pipeline
cost = pipeline.run()
print(f"Pipeline execution completed. Total cost: ${cost:.2f}")

[14:35:58] Checking operations...

           ✓ Operation 'summarize_prescriptions' (reduce)

           ✓ Operation 'resolve_medications' (resolve)

[14:35:59] ✓ Operation 'unnest_medications' (unnest)

           ✓ Operation 'extract_medications' (map)

...

           Cache flushed to disk.

           ╭─────── Step Execution: medical_info_extraction/boundary ────────╮                                     
           │ ✓ medical_info_extraction/scan_transcripts (Cost: $0.00)        │                                     
           │ ✓ medical_info_extraction/extract_medications (Cost: $0.02)     │                                     
           │ ✓ medical_info_extraction/unnest_medications (Cost: $0.00)      │                                     
           │ ✓ medical_info_extraction/resolve_medications (Cost: $0.01)     │                                     
           │ ✓ medical_info_extraction/summarize_prescriptions (Cost: $0.07) │                                     
           │ Step medical_info_extraction/boundary completed. Cost: $0.10    │                                     
           ╰─────────────────────────────────────────────────────────────────╯

           ✓ Saved to ./medication_summaries.json                                                                  
           

           ╭───────────────────────────────────────── Execution Summary ──────────────────────────────────────────╮
           │ Cost: $0.10                                                                                          │
           │ Time: 131.95s                                                                                        │
           │ Cache: intermediate_results                                                                          │
           │ Output: ./medication_summaries.json                                                                  │
           ╰──────────────────────────────────────────────────────────────────────────────────────────────────────╯

Pipeline execution completed. Total cost: $0.10
